In [ ]:
import inspect
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
from PIL import Image
import numpy as np


from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torch
from torch.nn import DataParallel
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torch.nn.functional")

from train_static import *

def get_activation(name):
    def hook(model, input, output):
        activations.append(output.detach().cpu())
    return hook

class CLIPVIT(nn.Module):
    def __init__(self, classnames, backbone_name='RN50', pos_embedding=False):
        super().__init__()

        self.num_clip = len(classnames)
        self.clip_model = load_clip_to_cpu(backbone_name)
        self.pos_embedding = pos_embedding

        # Disable gradients for all parameters first
        for param in self.clip_model.parameters():
            param.requires_grad = False
        
        # Tokenize all prompts at once and store them as a tensor
        self.tokenized_prompts = torch.stack([clip.tokenize(classname) for classname in classnames])

        # Register hooks to capture activations from each layer of the visual transformer
        for i in range(24):
            self.clip_model.visual.transformer.resblocks[i].register_forward_hook(get_activation(f'layer_{i}'))

    def forward(self, image):
        global activations
        activations = []  # Reset activations list for each forward pass

        if self.clip_model.training:
            self.clip_model.eval()

        # Move tokenized prompts to the same device as the input image
        tokenized_prompts = self.tokenized_prompts.to(image.device)

        # Process all tokenized prompts in a single forward pass
        self.clip_model(image, tokenized_prompts, self.pos_embedding)


        return activations

class ImageDataset(Dataset):
    def __init__(self, img_path):
        """
        img_path can be either a directory containing images or a path to a single image.
        """
        if os.path.isdir(img_path):
            self.img_dir = img_path
            self.image_names = [img for img in sorted(os.listdir(img_path))
                                if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
        elif os.path.isfile(img_path) and img_path.lower().endswith(('.png', '.jpg', '.jpeg')):
            self.img_dir, single_image_name = os.path.split(img_path)
            self.image_names = [single_image_name]  # Single image in a list
        else:
            raise ValueError(f"Provided path '{img_path}' is neither a directory nor a file, or file type is not supported.")

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.52997664, 0.48070561, 0.41943838],
                                 std=[0.27608301, 0.26593025, 0.28238822])
        ])

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, index):
        image_name = self.image_names[index]
        img_path = os.path.join(self.img_dir, image_name)
        image = Image.open(img_path).convert('RGB')  # Ensure image is RGB
        image = self.transform(image)
        
        return image_name, image
    
def run_model(model, data_loader, device=torch.device("cuda:0")):
    model.eval()
    model.to(device)
    
    progress_bar = tqdm(enumerate(data_loader), total=len(data_loader), desc=f"Getting activations")
    
    with torch.no_grad():
        for batch_idx, (batch_image_names, batch_images) in progress_bar:
            batch_images = batch_images.to(device)
            x = model(batch_images)
            
    
    return x


if __name__ == '__main__':
    
    ########################################
    img_dir = './Data/Things1854'
    lora = False
    load_hba = False
    n_dim = 66 # options: 49, 66
    backbone = 'ViT-L/14'
    batch_size = 1854
    ########################################

    if n_dim == 49:
        classnames = classnames49
    elif n_dim == 66:
        classnames = classnames66
    else: 
        raise ValueError("n_dim must be either 49 or 66")
    
    if backbone == 'RN50':
        pos_embedding = False
        print("pos_embedding is False")
    if backbone == 'ViT-B/16' or backbone == 'ViT-B/32' or backbone == 'ViT-L/14': 
        pos_embedding = True
        print("pos_embedding is True")

    model = CLIPVIT(classnames=classnames, backbone_name=backbone, pos_embedding=pos_embedding)
    
    # Load the dataset
    dataset = ImageDataset(img_dir)
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    # Run the model and save output embeddings
    activations = run_model(model, data_loader)
    torch.cuda.empty_cache()

In [ ]:
act_array = np.stack(activations, axis=0).transpose(0, 2, 1, 3)
act_array = act_array[:, :, 0, :]
act_array.shape

In [ ]:
np.save('./Encoder_Correspondence/CLIPViT_visual_encoder_24layers_activations.npy', act_array)

In [ ]:
import numpy as np
from scipy.spatial.distance import pdist, squareform
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

print("loading activations")
act_array = np.load('./Encoder_Correspondence/CLIPViT_visual_encoder_24layers_activations.npy')
print(f"Activation Array Shape: {act_array.shape}")  # Should print (24, 1854, 257, 1024)

def compute_rdm_cosine(act_array):
    num_layers, num_images, num_tokens, num_features = act_array.shape
    rdm_array = np.zeros((num_layers, num_images, num_images))

    for layer in tqdm(range(num_layers), desc="Computing RDM for layers"):
        flattened_vectors = act_array[layer].reshape(num_images, -1)  # Flatten the 257*1024 vectors
        rdm_layer = pdist(flattened_vectors, metric='cosine')  # Compute the pairwise cosine distance
        rdm_array[layer] = squareform(rdm_layer)  # Convert to a square matrix

    return rdm_array

def compute_rdm_cosine_with_sklearn(act_array):
    num_layers, num_images, num_features = act_array.shape
    rdm_array = np.zeros((num_layers, num_images, num_images))

    for layer in tqdm(range(num_layers), desc="Computing RDM for layers"):
        similarity_matrix = cosine_similarity(act_array[layer])  # Compute the pairwise cosine similarity
        rdm_array[layer] = 1 - similarity_matrix  # Convert similarity to distance

    return rdm_array

rdm_array = compute_rdm_cosine_with_sklearn(act_array)
print(f"Activation RDM array shape: {rdm_array.shape}")  # Should print (24, 1854, 1854)
save_path = './Encoder_Correspondence/CLIPViT_visual_encoder_24layers_RDM.npy'
np.save(save_path, rdm_array)
print(f"RDMs saved to {save_path}")


V2

In [ ]:
path = "./Encoder_Correspondence/CLIPViT_visual_encoder_24layers_optimal_weights.npy"
optimal_weights = np.load(path)
optimal_weights.shape

In [ ]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import pearsonr
from tqdm import tqdm

def weighted_rdm_combination(weights, layer_rdms):
    """Combine the RDMs with the given weights."""
    combined_rdm = np.sum(weights[:, np.newaxis, np.newaxis] * layer_rdms, axis=0)
    return combined_rdm

def objective_function(weights, layer_rdms, target_rdm):
    """Objective function to minimize (negative correlation)."""
    combined_rdm = weighted_rdm_combination(weights, layer_rdms)
    # Flatten the RDMs to compute the correlation
    combined_rdm_flat = combined_rdm[np.triu_indices(combined_rdm.shape[0], k=1)]
    target_rdm_flat = target_rdm[np.triu_indices(target_rdm.shape[0], k=1)]
    # Negative Pearson correlation
    return 1-pearsonr(combined_rdm_flat, target_rdm_flat)[0]

def find_optimal_weights(layer_rdms, target_rdm):
    """Find the optimal weights using constrained optimization."""
    n_layers = layer_rdms.shape[0]
    initial_weights = np.ones(n_layers) / n_layers  # Start with equal weights

    constraints = (
        {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},  # Sum of weights = 1
        {'type': 'ineq', 'fun': lambda w: w}  # Weights >= 0
    )

    result = minimize(
        objective_function, 
        initial_weights, 
        args=(layer_rdms, target_rdm), 
        method='SLSQP', 
        constraints=constraints
    )

    return result.x



weighting_matrix = np.zeros((281, 24))


# Initiate MEG RDMs
meg_rdm_path = "./Data/ThingsMEG_RDMs/THingsMEG_RDM_4P.npy"
meg_rdms_4p = np.load(meg_rdm_path)
meg_rdms = meg_rdms_4p[:-1, :, :, :] # use the first 3 participants, 4th data quality is not ideal
meg_rdms = np.mean(meg_rdms_4p, axis=0)

# Initiate layer_rdms
rdm_path = "./Encoder_Correspondence/CLIPViT_visual_encoder_24layers_RDM.npy"
layer_rdms = np.load(rdm_path)

for t in tqdm(range(281), desc="Optimizing"):  # Iterate over each timepoint
    target_rdm = meg_rdms[t]  # [1854, 1854] MEG RDM for timepoint t
    weights = find_optimal_weights(layer_rdms, target_rdm)
    weighting_matrix[t, :] = weights
    print(f"Timepoint {t}: {round(weights, 3)}")
